In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path
import logging
import pandas as pd
from datasets import Dataset
from flwr_datasets.partitioner import IidPartitioner
from datasets import Dataset, Sequence, Value, Features
from datasets import load_dataset

/home/alf/.pyenv/versions/tfg_flwr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fds = None
train_partitioner = None
def load_data(partition_id, num_partitions) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray] :
    # Download and partition dataset
    # Only initialize `FederatedDataset` once
    global fds
    global train_partitioner

    if fds is None or train_partitioner is None:
        train_partitioner = IidPartitioner(num_partitions=num_partitions)
        # Versión con path absoluto
        #data_files = context.run_config["data-path"] + "/" + context.run_config["data-set"]
        data_files : str = "data/buildingA-data.json"
        fds = load_dataset("json", data_files=data_files)
        train_partitioner.dataset =  fds["train"]
    
    train_dataset = train_partitioner.load_partition(partition_id)

    # Divide data on each node: 80% train, 20% test
    partition= train_dataset.train_test_split(test_size=0.2)

    partition["train"].set_format(type="numpy", columns=["features", "label"])
    partition["test"].set_format(type="numpy", columns=["features", "label"])

    x_train : np.ndarray = partition["train"][:]["features"].astype("float32")
    y_train : np.ndarray = partition["train"][:]["label"].astype("float32")
    x_test : np.ndarray = partition["test"][:]["features"].astype("float32")
    y_test : np.ndarray = partition["test"][:]["label"].astype("float32")

    return x_train, y_train, x_test, y_test

In [3]:
x_train, y_train, _,_ = load_data(0, 5)

In [ ]:
x_train = np.array(x_train)
shape = x_train.shape
x_train = x_train.reshape

(1216, 12, 21)